# Практическое занятие: Function Calling

**Цель:** научиться создавать инструменты (tools) для LLM, понять как работает Function Calling, и собрать простого агента.

**Что разберём:**
1. Function Calling — схема, вызов, параметр `tool_choice`
2. Несколько инструментов — поиск + калькулятор
3. Практика — агент-помощник (smolagents + Structured Output)

In [2]:
! pip install openai duckduckgo-search smolagents pydantic ddgs -q

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
authorizenet 1.1.6 requires lxml==4.*, but you have lxml 6.1.1 which is incompatible.
html-to-draftjs 1.0.1 requires beautifulsoup4<4.8,>4.7, but you have beautifulsoup4 4.15.0 which is incompatible.
label-studio 1.21.0 requires Django<5.2.0,>=5.1.8, but you have django 5.2.13 which is incompatible.
label-studio 1.21.0 requires django-filter==24.3, but you have django-filter 25.2 which is incompatible.
label-studio 1.21.0 requires openai<2.0.0,>=1.10.0, but you have openai 2.53.0 which is incompatible.
label-studio 1.21.0 requires pyarrow<19.0.0,>=18.1.0, but you have pyarrow 25.0.0 which is incompatible.
label-studio-ml 1.0.9 requires Jinja2==3.0.3, but you have jinja2 3.1.6 which is incompatible.
l

In [3]:
import json
from openai import OpenAI

BASE_URL = "http://localhost:11434/v1/"   
API_KEY = "ollama"
MODEL_NAME = "qwen3:8b"

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

---
## 1. Function Calling — как это работает

Определяем **схему** функции (JSON Schema). Модель **не вызывает** функцию сама — она возвращает **имя функции и аргументы**. Вызов делаем мы.

In [4]:
# --- Схема инструмента ---
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Получить текущую погоду в указанном городе",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Название города, например: Москва"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# --- Реальная функция (заглушка) ---
def get_weather(city: str) -> str:
    weather_data = {
        "Москва": "Облачно, +5°C",
        "Лондон": "Дождь, +12°C",
        "Токио": "Солнечно, +22°C",
    }
    return weather_data.get(city, f"Нет данных для города {city}")

print("Схема и функция определены.")

Схема и функция определены.


### Шаг 1: Отправляем запрос — смотрим что вернула модель

Модель **не вызывает** функцию. Она возвращает `tool_calls` с именем функции и аргументами.

In [5]:
messages = [{"role": "user", "content": "Какая погода в Москве?"}]

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    tools=tools,
)

assistant_message = response.choices[0].message
print("finish_reason:", response.choices[0].finish_reason)
print("tool_calls:", assistant_message.tool_calls)
print()

# Модель вернула аргументы — посмотрим на них
if assistant_message.tool_calls:
    tc = assistant_message.tool_calls[0]
    print(f"Функция: {tc.function.name}")
    print(f"Аргументы: {tc.function.arguments}")
    print(f"ID вызова: {tc.id}")
else:
    print("Модель не запросила вызов функции.")
    print("Ответ:", assistant_message.content)

finish_reason: tool_calls
tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_pwihgiu5', function=Function(arguments='{"city":"Москва"}', name='get_weather'), type='function', index=0)]

Функция: get_weather
Аргументы: {"city":"Москва"}
ID вызова: call_pwihgiu5


### Шаг 2: Вызываем функцию сами и отправляем результат обратно

In [6]:
# Вызываем функцию САМИ — модель только дала аргументы
tc = assistant_message.tool_calls[0]
args = json.loads(tc.function.arguments)
result = get_weather(**args)
print(f"Результат вызова get_weather({args}): {result}")

# Отправляем результат обратно модели
messages.append(assistant_message)                     # ответ ассистента с tool_calls
messages.append({
    "role": "tool",
    "tool_call_id": tc.id,
    "content": result,
})

final_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
)

print("\nФинальный ответ модели:")
print(final_response.choices[0].message.content)

Результат вызова get_weather({'city': 'Москва'}): Облачно, +5°C

Финальный ответ модели:
Сейчас в Москве облачно, температура воздуха составляет +5°C. Погода немного прохладная, но комфортная для прогулок!


---
## 2. Несколько инструментов через smolagents: Поиск + Калькулятор

Используем готовый `DuckDuckGoSearchTool` из smolagents + свой `@tool` калькулятор. Агент сам выбирает какой инструмент вызвать.

In [7]:
from smolagents import tool, CodeAgent, DuckDuckGoSearchTool, OpenAIServerModel

# Готовый инструмент поиска из smolagents
search_tool = DuckDuckGoSearchTool()

# Свой инструмент — калькулятор
@tool
def calculator(expression: str) -> str:
    """Вычисляет математическое выражение и возвращает результат.
    
    Args:
        expression: математическое выражение, например '(15 + 27) * 3 - 12'
    """
    allowed = set("0123456789+-*/(). ")
    if not all(c in allowed for c in expression):
        return "Ошибка: недопустимые символы"
    return str(eval(expression))

# Создаём агента с двумя инструментами
model = OpenAIServerModel(model_id=MODEL_NAME, api_base=BASE_URL, api_key=API_KEY)
agent = CodeAgent(tools=[search_tool, calculator], model=model)

print("Агент готов: DuckDuckGoSearchTool + calculator")

Агент готов: DuckDuckGoSearchTool + calculator


### Запрос 1: Поиск

In [8]:
result = agent.run("Кто выиграл последний чемпионат мира по футболу?")
print("\n=== Ответ ===")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Кто выиграл последний чемпионат мира по футболу?                                                                │
│                                                                                                                 │
╰─ OpenAIModel - qwen3:8b ────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  last_world_cup_winner = web_search(query="winner of the last FIFA World Cup")                                    
  print(last_world_cup_winner)                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[List of FIFA World Cup finals - Wikipedia](https://en.wikipedia.org/wiki/List_of_FIFA_World_Cup_finals)
The FIFA World Cup is an international association football competition contested by the senior men's national 
teams of the Fédération Internationale de Football Association, the sport's global governing body.

[FIFA World Cup Winners List (1930 to 2026), Check Winning 
Teams...](https://www.careerpower.in/fifa-world-cup-winners-list.html)
If you are preparing for SSC, railway, and other government exams, you have to be aware of the FIFA World Cup 2026 
winners and other key points that can be asked in the exam as part of current affairs and general awareness.The 
last FIFA World Cup 2018 was won by France.

[FIFA Men's World Cup Winners List | FOX Sports](https://www.foxsports.com/soccer/fifa-world-cup/history)
From countries who won the FIFA World Cup to leading goal scorers, our comprehensive history guide has everything 
you need.

[Five Footballers That Would Likely Play in their Last FIFA World 
Cup...](https://4howtodo.com/five-footballers-that-would-likely-play-in-their-last-fifa-world-cup-this-year/)
The 2022 FIFA World is getting closer, and we’re set to see the first game start. It is gearing to be one of the 
best FIFA World Cups, and we can’t wait to.

[Argentina commemorate win against England at World Cup 2026 
with...](https://www.theguardian.com/football/2026/aug/06/argentinas-fa-commemorate-world-cup-semi-final-win-englan
d)
Argentina’s World Cup semi-final win over England last month is being commemorated by the country’s football 
association. The Argentinian football association (AFA) has established 15 July – the date of the 2-1 win in 
Atlanta – as its ‘National Football Teams Day’...

[FIFA World Cup 2026: Live Scores, Schedule, Tickets & Travel](https://fifa-worldcup26.com/)
The complete World Cup 2026 hub. Get live scores, standings, and match schedules. Plus, official ticket updates, 
travel tips, and broadcasting guides.

[Watch: Argentina's FIFA World Cup winner Emiliano... - Times of 
India](https://timesofindia.indiatimes.com/sports/football/top-stories/watch-argentinas-fifa-world-cup-winner-emili
ano-martinez-inaugurates-mohun-bagans-pele-maradona-sobers-gate/articleshow/101508408.cms)
Football News: Emiliano Martinez, the celebrated goalkeeper who played a crucial role in Argentina's FIFA World Cup
win in Qatar, inaugurated the Pele-Maradona-Sober...

[FIFA World Cup final 2026 highlights: Spain outlasts Argentina 1-0 
to...](https://www.nbcnews.com/sports/soccer/live-blog/fifa-world-cup-final-2026-argentina-spain-messi-yamal-live-u
pdates-rcna587754)
The last World Cup final to take place in the U.S. ended in a 0-0 draw after extra time.Maybe the real winner of 
the 2026 World Cup is Lamine Yamal’s 3-year-old brother, Keyne. Longtime fans of the Barcelona FC star know that 
wherever Yamal goes, a small shadow is sure to follow.

[Qatar World Cup 2022 : Qatar 2022 FIFA World Cup... | 
emiilythenoob](https://emiilythenoob.blogspot.com/2021/02/qatar-world-cup-2022-qatar-2022-fifa.html)
There has now been a european winner in the last four world cups: England meet poland, france face ukraine, and 
spain will tackle sweden in qualifying for the 2022 world cup in qatar. The 2022 fifa world cup (arabic...

[Post-world Cup Depression: Factors Behind... - Al-Estiklal 
Newspaper](https://www.alestiklal.net/en/article/post-world-cup-depression-factors-behind-the-disturbing-distress-e
xperienced-by-football-fans-at-the-end-of-the-world-cup)
Experts and mental health specialists warn that many of the World Cup fans might be exposed to long lasting periods
of sadness.

Out: None

[Step 1: Duration 16.03 seconds| Input tokens: 2,153 | Output tokens: 355]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Argentina")                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Argentina

[Step 2: Duration 13.65 seconds| Input tokens: 5,419 | Output tokens: 834]


=== Ответ ===
Argentina


### Запрос 2: Калькулятор

In [9]:
result = agent.run("Сколько будет (15 + 27) * 3 - 12?")
print("\n=== Ответ ===")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Сколько будет (15 + 27) * 3 - 12?                                                                               │
│                                                                                                                 │
╰─ OpenAIModel - qwen3:8b ────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = calculator(expression="(15 + 27) * 3 - 12")                                                             
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 114

[Step 1: Duration 9.48 seconds| Input tokens: 2,156 | Output tokens: 340]


=== Ответ ===
114


---
## 3. Практическая работа: Агент-помощник магазина

**Задание:** Напишите инструменты для агента, который консультирует по товарам.

Ниже — каталог товаров. Ваша задача:
1. Написать `@tool` функции для smolagents
2. Создать агента и проверить на запросах

In [10]:
# Каталог товаров: название -> описание
CATALOG = {
    "iPhone 15": "Смартфон Apple, 128 ГБ, камера 48 Мп, цена 89 990 руб.",
    "Samsung Galaxy S24": "Смартфон Samsung, 256 ГБ, AI-функции, цена 74 990 руб.",
    "MacBook Air M3": "Ноутбук Apple, чип M3, 16 ГБ RAM, цена 124 990 руб.",
    "Lenovo ThinkPad": "Ноутбук для работы, 16 ГБ RAM, цена 67 990 руб. Нет в наличии.",
    "AirPods Pro 2": "Наушники Apple с шумоподавлением, цена 19 990 руб.",
    "Sony WH-1000XM5": "Полноразмерные наушники Sony, цена 24 990 руб.",
    "iPad Air": "Планшет Apple, чип M2, 10.9 дюймов, цена 59 990 руб.",
    "Samsung Tab S9": "Планшет Samsung, AMOLED, цена 49 990 руб. Нет в наличии.",
    "Apple Watch SE": "Умные часы Apple, цена 29 990 руб.",
    "Xiaomi Band 8": "Фитнес-браслет Xiaomi, цена 3 990 руб.",
}

print(f"В каталоге {len(CATALOG)} товаров")

В каталоге 10 товаров


---
### Вариант A: smolagents

Напишите `@tool` функцию, которая ищет товары в `CATALOG` по подстроке. Создайте `CodeAgent`.


Подсказка - если есть вопросы по работе с smolagents - гуглите

In [11]:
from smolagents import tool, CodeAgent, OpenAIServerModel

def search_catalog(query: str) -> str:
    # Опишите docstring функции
    # реализуйте поиск
    pass

# Создайте модель и агента (смотрите пример выше)
# model = ...
# agent = ...

### Запускаем и смотрим как агент работает

In [12]:
# Проверьте агента:
# result = agent.run("Какие наушники есть в магазине?")
# print(result)

In [13]:
# Ещё запросы для проверки:
# result = agent.run("Есть ли у вас что-то от Apple дешевле 30 000?")
# print(result)

---
### Вариант B: Structured Output (Pydantic + OpenAI SDK)

Та же задача — поиск по каталогу, но модель возвращает структурированный JSON с запросом для поиска.

In [14]:
from pydantic import BaseModel

# Определите модель: модель должна вернуть список поисковых запросов к каталогу
class SearchPlan(BaseModel):
    # список товаров и не забудьте reasoning
    pass

In [15]:
def run_structured_agent(user_query: str):
    # Шаг 1: Получите SearchPlan от модели через client.beta.chat.completions.parse()
    # Подсказка: response_format
    
    # Шаг 2: Для каждого query из плана — найдите товары в CATALOG (как в search_catalog)
    
    # Шаг 3: Отправьте результаты модели для финального ответа текстом
    
    pass

### Запускаем Structured Output агента

In [16]:
# Проверьте:
# answer = run_structured_agent("Какие наушники есть в магазине?")
# print(answer)

In [17]:
# Ещё запросы для проверки:
# answer = run_structured_agent("Есть ли у вас что-то от Apple дешевле 30 000?")
# print(answer)

In [18]:
from smolagents import tool, CodeAgent, OpenAIServerModel

@tool
def search_catalog(query: str) -> str:
    """Ищет товары в каталоге магазина по подстроке в названии или описании.
    Используй эту функцию, когда пользователь спрашивает про товары определённой
    категории, бренда, или с определённым ценовым ограничением.

    Args:
        query: поисковый запрос — часть названия товара, бренд или ключевое слово
               (например: "наушники", "Apple", "Samsung")
    """
    # приводим запрос к нижнему регистру для нечувствительного к регистру поиска,
    # тот же приём, что и в get_weather/search_product ранее
    query = query.lower()

    # ищем совпадения query И в названии, И в описании товара —
    # так "Apple" найдёт и "iPhone 15" (Apple в описании), и "AirPods Pro 2"
    matches = [
        f"{name}: {description}"
        for name, description in CATALOG.items()
        if query in name.lower() or query in description.lower()
    ]

    # если ничего не нашли — возвращаем текст, а не пустую строку/исключение,
    # чтобы агент мог осмысленно среагировать в своём рассуждении
    if not matches:
        return f"По запросу '{query}' ничего не найдено"

    # склеиваем все найденные товары построчно — модели удобнее читать список
    return "\n".join(matches)


# Модель — обёртка smolagents над OpenAI-совместимым API (та же логика,
# что и ChatOpenAI/OpenAI(client) ранее, просто другой SDK)
model = OpenAIServerModel(model_id=MODEL_NAME, api_base=BASE_URL, api_key=API_KEY)

# CodeAgent — агент, который для использования инструментов пишет и исполняет
# Python-код (в отличие от create_react_agent/tool_calls, где вызовы идут через JSON) —
# именно поэтому у search_catalog должна быть обычная функция с @tool, а не JSON-схема
agent = CodeAgent(tools=[search_catalog], model=model)

In [19]:
# Проверка
result = agent.run("Какие наушники есть в магазине?")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Какие наушники есть в магазине?                                                                                 │
│                                                                                                                 │
╰─ OpenAIModel - qwen3:8b ────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  headphones_list = search_catalog(query="наушники")                                                               
  final_answer(headphones_list)                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: AirPods Pro 2: Наушники Apple с шумоподавлением, цена 19 990 руб.
Sony WH-1000XM5: Полноразмерные наушники Sony, цена 24 990 руб.

[Step 1: Duration 10.25 seconds| Input tokens: 2,162 | Output tokens: 280]

AirPods Pro 2: Наушники Apple с шумоподавлением, цена 19 990 руб.
Sony WH-1000XM5: Полноразмерные наушники Sony, цена 24 990 руб.


In [20]:
result = agent.run("Есть ли у вас что-то от Apple дешевле 30 000?")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Есть ли у вас что-то от Apple дешевле 30 000?                                                                   │
│                                                                                                                 │
╰─ OpenAIModel - qwen3:8b ────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = search_catalog(query='Apple price less than 30000')                                                     
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
По запросу 'apple price less than 30000' ничего не найдено

Out: None

[Step 1: Duration 12.70 seconds| Input tokens: 2,171 | Output tokens: 443]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (4 outputs left out)

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Да, есть: AirPods Pro 2 (19 990 руб.) и Apple Watch SE (29 990 руб.).")                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Да, есть: AirPods Pro 2 (19 990 руб.) и Apple Watch SE (29 990 руб.).

[Step 3: Duration 9.21 seconds| Input tokens: 7,142 | Output tokens: 1,059]

Да, есть: AirPods Pro 2 (19 990 руб.) и Apple Watch SE (29 990 руб.).


In [21]:
from pydantic import BaseModel, Field
from typing import List

class SearchPlan(BaseModel):
    """План поиска: список запросов к каталогу для ответа на вопрос пользователя"""

    # reasoning первым — та же логика, что мы разбирали на MatchResult:
    # модель должна сначала обдумать, ЧТО именно нужно искать,
    # и только потом сформировать список конкретных запросов
    reasoning: str = Field(
        description="Краткое рассуждение о том, какая информация нужна, "
                    "чтобы ответить на вопрос пользователя, и почему выбраны именно эти запросы"
    )

    # список отдельных поисковых запросов — не один общий, а разбитый на части,
    # чтобы каждый можно было по отдельности прогнать через search_catalog
    queries: List[str] = Field(
        description="Список коротких поисковых запросов к каталогу товаров "
                    "(например, бренды, категории, ключевые слова из вопроса пользователя)"
    )

In [22]:
def run_structured_agent(user_query: str):
    """Двухшаговый агент без цикла: план поиска -> ручной поиск -> финальный ответ"""

    # === Шаг 1: просим модель составить план поиска по Pydantic-схеме ===
    plan_response = client.beta.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "Ты — консультант магазина электроники. По вопросу пользователя "
                           "составь список поисковых запросов к каталогу товаров, "
                           "которые помогут собрать нужную информацию для ответа."
            },
            {"role": "user", "content": user_query},
        ],
        response_format=SearchPlan,
    )

    plan = plan_response.choices[0].message.parsed

    # если модель отказалась/не смогла сформировать план — защищаемся,
    # как обсуждали раньше с .refusal
    if plan is None:
        return "Не удалось спланировать поиск по запросу."

    # === Шаг 2: сами (без модели!) выполняем поиск по каждому query из плана ===
    # это НЕ вызов LLM — обычная Python-функция, копия логики search_catalog,
    # только без декоратора @tool, потому что здесь нет агента, который бы её вызывал
    search_results = []
    for query in plan.queries:
        query_lower = query.lower()
        matches = [
            f"{name}: {description}"
            for name, description in CATALOG.items()
            if query_lower in name.lower() or query_lower in description.lower()
        ]
        found_text = "\n".join(matches) if matches else f"По запросу '{query}' ничего не найдено"
        search_results.append(f"Запрос '{query}':\n{found_text}")

    # склеиваем результаты всех запросов в один текстовый блок для модели
    all_results_text = "\n\n".join(search_results)

    # === Шаг 3: отдаём собранные данные модели, просим финальный текстовый ответ ===
    final_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "Ты — консультант магазина электроники. "
                           "На основе результатов поиска по каталогу дай развёрнутый ответ "
                           "пользователю на русском языке."
            },
            {
                "role": "user",
                "content": f"Вопрос пользователя: {user_query}\n\n"
                           f"Результаты поиска по каталогу:\n{all_results_text}"
            },
        ],
    )

    return final_response.choices[0].message.content

In [23]:
# Проверка
answer = run_structured_agent("Какие наушники есть в магазине?")
print(answer)

В магазине доступны следующие модели наушников:

1. **AirPods Pro 2**  
   - Производитель: Apple  
   - Основные особенности: шумоподавление, беспроводное соединение, совместимость с экосистемой Apple.  
   - Цена: 19 990 рублей.  

2. **Sony WH-1000XM5**  
   - Производитель: Sony  
   - Основные особенности: беспроводные, активное шумоподавление, высококачественное звучание, длительный срок работы от батареи.  
   - Цена: 24 990 рублей.  

К сожалению, по другим запросам (например, проводные наушники, водонепроницаемые модели, наушники с креплением на ухо и др.) в каталоге ничего не найдено. Если вас интересуют определённые характеристики или бренды, уточните, и я постараюсь помочь!


In [24]:
answer = run_structured_agent("Есть ли у вас что-то от Apple дешевле 30 000?")
print(answer)

На данный момент в нашем каталоге не найдено товаров бренда Apple с ценой ниже 30 000 рублей. Это касается всех популярных моделей, включая iPhone, iPad, MacBook, Apple Watch, AirPods и другие аксессуары. Возможно, товары временно отсутствуют в наличии или их стоимость превышает указанную сумму. 

Рекомендуем уточнить детали запроса (например, конкретную модель или цену) или посетить сайт магазина для проверки актуального ассортимента. Если у вас есть другие пожелания — с радостью помогу найти подходящие решения!
